In [0]:
%run /Configurations/Init_Scripts

In [0]:
import os
from datetime import datetime
import shutil 
import pandas as pd
import traceback

In [0]:
dbutils.widgets.text("InvoiceDate", "")
InvoiceDate = dbutils.widgets.get("InvoiceDate")

dbutils.widgets.text("DateInterval", "1")
DateInterval = dbutils.widgets.get("DateInterval")

assertion_errors = []

TestCaseDetails = {"validateInvoiceFiles": {
                        "ErrorMessage": "Invoice Files are not generated properly",
                        "TestCaseDescription": "Validate Invoice Files are generated in Azure Storage Account"
                    },
                    "validateRevenueAccrualFiles": {
                        "ErrorMessage": "Data discrepancy between the Revenue Accrual File and Goldzone.InvoiceAddendum_v",
                        "TestCaseDescription": "Validate that the Revenue Accrual File matches with the Goldzone.InvoiceAddendum_v"
                    }
                    }

In [0]:
if InvoiceDate == '':
    InvoiceDate  = (spark.read.table('promotion.dim_invoicecyclemonth')
                          .filter(col('InvoiceDate').between(date_sub(current_date(), lit(DateInterval)),date_add(current_date(), lit(DateInterval))))
                          .select('InvoiceDate').orderBy(desc('InvoiceDate'))
                          .first()[0]
                          )

print(f"InvoiceDate:{InvoiceDate}")

In [0]:
#Varaiable
InvoiceDate = str(InvoiceDate)

current_month = InvoiceDate.split('-')[1]
current_day = InvoiceDate.split('-')[2]
current_year = InvoiceDate.split('-')[0]

AccrualFileName = f"Per Patient Pricing Revenue Accrual File {current_month}_{current_year}.xlsx"

temp_path = '/tmp/'+AccrualFileName

accrualsheet_dict = [{"SheetName":"Accrual File Details",
                     "HashCols":['Total Invoices Amount','Total Per Patient Fees','Total Per Patient Fees Exception Pre-Invoice','Total Per Patient Fees Invoiced','Total Per Cycle Fees','Total Per Cycle Fees Exception Pre-Invoice','Total Per Cycle Fees Invoiced','Total No Patient Association Fees','Total No Patient Association Fees Pre-Invoice','Total No Patient Association Fees Invoiced','Total Per Cycle Exception Fees','Total Post Invoice Exception Amount','Total Per Patient Fee MapViolation','Total NonP3 Patient FraudViolation','Total NonP3 Fee'],
                     "Schema":StructType([
									StructField("Ship To", LongType(), True),
									StructField("Total Invoices Amount", LongType(), True),
									StructField("Total Per Patient Fees", LongType(), True),
									StructField("Total Per Patient Fees Exception Pre-Invoice", LongType(), True),
									StructField("Total Per Patient Fees Invoiced", LongType(), True),
         							StructField("Total Per Cycle Fees", LongType(), True),
									StructField("Total Per Cycle Fees Exception Pre-Invoice", LongType(), True),
									StructField("Total Per Cycle Fees Invoiced", LongType(), True),
									StructField("Total No Patient Association Fees", LongType(), True),
									StructField("Total No Patient Association Fees Pre-Invoice", LongType(), True),
									StructField("Total No Patient Association Fees Invoiced", LongType(), True),
									StructField("Total Per Cycle Exception Fees", LongType(), True),
									StructField("Total Post Invoice Exception Amount", LongType(), True),
									StructField("Total Per Patient Fee MapViolation", LongType(), True),
									StructField("Total NonP3 Patient FraudViolation", LongType(), True),
									StructField("Total NonP3 Fee", LongType(), True),
									StructField("Billing Period Start Date", StringType(), True),
									StructField("Billing Period End Date", StringType(), True),
									StructField("Invoice Calculation Date", StringType(), True)
								])},
                     {"SheetName":"Accrual File Summary",
                     "HashCols":[''],
					 "Schema": StructType([
								StructField("Ship To", LongType(), True),
								StructField("Revenue Accrual Description", StringType(), True),
								StructField("Revenue Accrual Definition", StringType(), True),
								StructField("Revenue Accrual Total", LongType(), True),
								StructField("Billing Period Start Date", StringType(), True),
								StructField("Billing Period End Date", StringType(), True),
								StructField("Invoice Calculation Date", StringType(), True)
							])},
                     {"SheetName":"Accrual File Midway Details",
                     "HashCols":['Total MidwayPatient Fee'],
					 "Schema": StructType([
							StructField("Ship To", LongType(), True),
							StructField("Total MidwayPatient Fee", LongType(), True),
							StructField("Billing Period Start Date", StringType(), True),
							StructField("Billing Period End Date", StringType(), True),
							StructField("Invoice Calculation Date", StringType(), True)
						])},
                     {"SheetName":"Accrual File Midway Summary",
                     "HashCols":[''],
					 "Schema":StructType([
									StructField("Ship To", LongType(), True),
									StructField("Revenue Accrual Description", StringType(), True),
									StructField("Revenue Accrual Definition", StringType(), True),
									StructField("Revenue Accrual Total", LongType(), True),
									StructField("Billing Period Start Date", StringType(), True),
									StructField("Billing Period End Date", StringType(), True),
									StructField("Invoice Calculation Date", StringType(), True)
								])}
                     ]

In [0]:
def fetchInvoiceAddendumView(InvoiceDate,PromotionName):
    df_AddendumView = spark.sql(f"""
            Select 
                regexp_replace(ShipToAccountId, '^[0]*', '') as `Ship To`,
                PatientClassificationName as `Material Description`,
                PatientClassificationOrder,
                coalesce(TotalCycleCount,0) as `Total Cycles`,
                coalesce(TotalVisitCount,0) as `QTY`,
                coalesce(InvoiceAmount ,0)as `Total`,
                coalesce(InvoiceCalculationDate,'{InvoiceDate}')  as InvoiceCalculationDate,
                BillingPeriodStartDate,
                BillingPeriodEndDate
            From goldzone.invoiceaddendum_v
            where InvoiceCalculationDate = cast('{InvoiceDate}' as date) and EffectiveDate <= cast('{InvoiceDate}' as date) and TerminationDate  >=  add_months(cast('{InvoiceDate}' as date),-1 ) and PromotionName = '{PromotionName}'
            UNION
            Select 
                regexp_replace(DA.ShipToAccountId, '^[0]*', '') as `Ship To`,
                PatientClassificationName as `Material Description`,
                PatientClassificationOrder,
                coalesce(TotalCycleCount,0) as `Total Cycles`,
                coalesce(TotalVisitCount,0) as `QTY`,
                coalesce(InvoiceAmount ,0)as `Total`,
                coalesce(InvoiceCalculationDate,'{InvoiceDate}')  as InvoiceCalculationDate,
                ICM.CalendarStartTimeStamp AS BillingPeriodStartDate,
                ICM.CalendarEndTimeStamp AS BillingPeriodEndDate
            From promotion.dim_account DA
            INNER JOIN promotion.dim_promotion PM 
            ON DA.PromotionUUID = PM.PromotionUUID and PM.PromotionName = '{PromotionName}'
            LEFT JOIN goldzone.invoiceaddendum_v FUL ON FUL.ShipToAccountUUID=DA.ShipToAccountUUID AND FUL.InvoiceCalculationDate= cast('{InvoiceDate}' as date)
            LEFT JOIN promotion.dim_invoicecyclemonth  AS ICM  ON coalesce(FUL.InvoiceCalculationDate,'{InvoiceDate}') = cast(ICM.InvoiceDate as date)
            where DA.EffectiveDate <= cast('{InvoiceDate}' as date) and DA.TerminationDate  >=  add_months(cast('{InvoiceDate}' as date),-1 )
            """)

    return df_AddendumView

In [0]:
def validateAccrualDetailsSheet(df_AccrualExcelSheet,InvoiceDate,PromotionName,HashCols):
    
    df_AccrualFile = df_AccrualExcelSheet.withColumn("HashKey",sha2(concat_ws('', *HashCols),256))

    df_AddendumView = fetchInvoiceAddendumView(InvoiceDate,PromotionName)

    df_AccuralDetails = (df_AddendumView.groupBy("Ship To","InvoiceCalculationDate","BillingPeriodStartDate","BillingPeriodEndDate").agg(
                                            coalesce(sum(when(col("PatientClassificationOrder").isin(1,2,3,4,5,6,7,12,13,14),col("Total"))),lit('0')).cast(LongType()).alias("Total Invoices Amount"),
                                            coalesce(sum(when(col("PatientClassificationOrder")==1,col("Total"))), lit('0')).cast(LongType()).alias("Total Per Patient Fees"),
                                            coalesce(sum(when(col("PatientClassificationOrder")==2,col("Total"))),lit('0')).cast(LongType()).alias("Total Per Patient Fees Exception Pre-Invoice"),
                                            coalesce(sum(when(col("PatientClassificationOrder").isin(1,2),col("Total"))),lit('0')).cast(LongType()).alias("Total Per Patient Fees Invoiced"),
                                            coalesce(sum(when(col("PatientClassificationOrder")==3,col("Total"))),lit('0')).cast(LongType()).alias("Total Per Cycle Fees"),
                                            coalesce(sum(when(col("PatientClassificationOrder")==4,col("Total"))),lit('0')).cast(LongType()).alias("Total Per Cycle Fees Exception Pre-Invoice"),
                                            coalesce(sum(when(col("PatientClassificationOrder").isin(3,4),col("Total"))),lit('0')).cast(LongType()).alias("Total Per Cycle Fees Invoiced"),
                                            coalesce(sum(when(col("PatientClassificationOrder")==5,col("Total"))), lit('0')).cast(LongType()).alias("Total No Patient Association Fees"),
                                            coalesce(sum(when(col("PatientClassificationOrder")==6,col("Total"))),lit('0')).cast(LongType()).alias("Total No Patient Association Fees Pre-Invoice"),
                                            coalesce(sum(when(col("PatientClassificationOrder").isin(5,6),col("Total"))),lit('0')).cast(LongType()).alias("Total No Patient Association Fees Invoiced"),
                                            coalesce(sum(when(col("PatientClassificationOrder")==7,col("Total"))), lit('0')).cast(LongType()).alias("Total Per Cycle Exception Fees"),
                                            coalesce(sum(when(col("PatientClassificationOrder").isin(9,10,11),col("Total"))),lit('0')).cast(LongType()).alias("Total Post Invoice Exception Amount"),
                                            coalesce(sum(when(col("PatientClassificationOrder")==12,col("Total"))),lit('0')).cast(LongType()).alias("Total Per Patient Fee MapViolation"),
                                            coalesce(sum(when(col("PatientClassificationOrder")==13,col("Total"))),lit('0')).cast(LongType()).alias("Total NonP3 Patient FraudViolation"),
                                            coalesce(sum(when(col("PatientClassificationOrder")==14,col("Total"))),lit('0')).cast(LongType()).alias("Total NonP3 Fee"),
                                            date_format(col("BillingPeriodStartDate"),'MM-dd-yyyy').alias("Billing Period Start Date"),
                                            date_format(col("BillingPeriodEndDate"),'MM-dd-yyyy').alias("Billing Period End Date"),
                                            date_format(col("InvoiceCalculationDate"),'MM-dd-yyyy').alias("Invoice Calculation Date")
                                            ).drop("InvoiceCalculationDate","BillingPeriodStartDate","BillingPeriodEndDate", "PatientClassificationOrder")
                                            .withColumn("HashKey",sha2(concat_ws('', *HashCols),256)))
    
    df_AccrualFile = add_Prefix_ColumnName(df_AccrualFile,"AccrualFile_")
    df_AccuralDetails = add_Prefix_ColumnName(df_AccuralDetails,"AddendumView_")
    
    df_result = (df_AccrualFile.alias("file").join(df_AccuralDetails.alias("goldzone"),(col("file.AccrualFile_Ship To") == col("goldzone.AddendumView_Ship To")),"FULL")
                                                    .withColumn("Comments",when(col("file.AccrualFile_HashKey") == col("goldzone.AddendumView_HashKey"),"Matched").otherwise("Unknown"))) 
    
    return df_result

In [0]:
def validateAccrualSummarySheet(df_AccrualExcelSheet,InvoiceDate,PromotionName,schema):

    df_AddendumView = fetchInvoiceAddendumView(InvoiceDate,PromotionName)

    df_AccuralSummary = (df_AddendumView.groupBy("InvoiceCalculationDate","BillingPeriodStartDate","BillingPeriodEndDate").agg(
                                countDistinct(col("Ship To")).alias("Ship To"),
                                coalesce(sum(when(col("PatientClassificationOrder").isin(1,2,3,4,5,6,7,12,13,14),col("Total"))),lit('0')).cast(LongType()).alias("Total Invoices Amount"),
                                coalesce(sum(when(col("PatientClassificationOrder")==1,col("Total"))),lit('0')).cast(LongType()).alias("Total Per Patient Fees"),
                                coalesce(sum(when(col("PatientClassificationOrder")==2,col("Total"))),lit('0')).cast(LongType()).alias("Total Per Patient Fees Exception Pre-Invoice"),
                                coalesce(sum(when(col("PatientClassificationOrder").isin(1,2),col("Total"))),lit('0')).cast(LongType()).alias("Total Per Patient Fees Invoiced"),
                                coalesce(sum(when(col("PatientClassificationOrder")==3,col("Total"))),lit('0')).cast(LongType()).alias("Total Per Cycle Fees"),
                                coalesce(sum(when(col("PatientClassificationOrder")==4,col("Total"))),lit('0')).cast(LongType()).alias("Total Per Cycle Fees Exception Pre-Invoice"),
                                coalesce(sum(when(col("PatientClassificationOrder").isin(3,4),col("Total"))),lit('0')).cast(LongType()).alias("Total Per Cycle Fees Invoiced"),
                                coalesce(sum(when(col("PatientClassificationOrder")==5,col("Total"))),lit('0')).cast(LongType()).alias("Total No Patient Association Fees"),
                                coalesce(sum(when(col("PatientClassificationOrder")==6,col("Total"))),lit('0')).cast(LongType()).alias("Total No Patient Association Fees Pre-Invoice"),
                                coalesce(sum(when(col("PatientClassificationOrder").isin(5,6),col("Total"))),lit('0')).cast(LongType()).alias("Total No Patient Association Fees Invoiced"),
                                coalesce(sum(when(col("PatientClassificationOrder")==7,col("Total"))),lit('0')).cast(LongType()).alias("Total Per Cycle Exception Fees"),
                                coalesce(sum(when(col("PatientClassificationOrder").isin(9,10,11),col("Total"))),lit('0')).cast(LongType()).alias("Total Post Invoice Exception Amount"),  
                                coalesce(sum(when(col("PatientClassificationOrder")==12,col("Total"))),lit('0')).cast(LongType()).alias("Total Per Patient Fee MapViolation"),
                                coalesce(sum(when(col("PatientClassificationOrder")==13,col("Total"))),lit('0')).cast(LongType()).alias("Total NonP3 Patient FraudViolation"),
                                coalesce(sum(when(col("PatientClassificationOrder")==14,col("Total"))),lit('0')).cast(LongType()).alias("Total NonP3 Fee"),                                
                                date_format(col("BillingPeriodStartDate"),'MM-dd-yyyy').alias("Billing Period Start Date"),
                                date_format(col("BillingPeriodEndDate"),'MM-dd-yyyy').alias("Billing Period End Date"),
                                date_format(col("InvoiceCalculationDate"),'MM-dd-yyyy').alias("Invoice Calculation Date")
                                ).drop("InvoiceCalculationDate","BillingPeriodStartDate","BillingPeriodEndDate", "PatientClassificationOrder")
                                )

    pandasDf = df_AccuralSummary.toPandas()
    pandasDf = pandasDf.melt(id_vars=["Ship To","Billing Period Start Date","Billing Period End Date","Invoice Calculation Date"], var_name="Revenue Accrual Description", value_name="Revenue Accrual Total")
    pandasDf.insert(1, "Revenue Accrual Definition", "")
    pandasDf = pandasDf[["Ship To","Revenue Accrual Description","Revenue Accrual Definition","Revenue Accrual Total","Billing Period Start Date","Billing Period End Date","Invoice Calculation Date"]]

    df_AccuralSummary = spark.createDataFrame(pandasDf,schema)

    df_AccrualExcelSheet = add_Prefix_ColumnName(df_AccrualExcelSheet,"AccrualFile_")
    df_AccuralSummary = add_Prefix_ColumnName(df_AccuralSummary,"AddendumView_")

    df_result = (df_AccrualExcelSheet.alias("file").join(df_AccuralSummary.alias("goldzone"),(col("file.AccrualFile_Invoice Calculation Date") == col("goldzone.AddendumView_Invoice Calculation Date")) & (col("file.AccrualFile_Revenue Accrual Description") == col("goldzone.AddendumView_Revenue Accrual Description")),"FULL")
                                                    .withColumn("Comments",when(col("file.AccrualFile_Revenue Accrual Total") == col("goldzone.AddendumView_Revenue Accrual Total"),"Matched").otherwise("Unknown"))) 
    
    return df_result


In [0]:
def validateMidwaySummarySheet(df_AccrualExcelSheet,InvoiceDate,PromotionName,schema):

    df_AddendumView = fetchInvoiceAddendumView(InvoiceDate,PromotionName)

    df_AccuralMidwaySummary = (df_AddendumView.filter(col("PatientClassificationOrder")==15).groupBy("InvoiceCalculationDate","BillingPeriodStartDate","BillingPeriodEndDate").agg(
							countDistinct(col("Ship To")).alias("Ship To"),
							coalesce(sum(when(col("PatientClassificationOrder")==15,col("Total"))),lit('0')).cast(LongType()).alias("Total MidwayPatient Fees"),                                
							date_format(col("BillingPeriodStartDate"),'MM-dd-yyyy').alias("Billing Period Start Date"),
							date_format(col("BillingPeriodEndDate"),'MM-dd-yyyy').alias("Billing Period End Date"),
							date_format(col("InvoiceCalculationDate"),'MM-dd-yyyy').alias("Invoice Calculation Date")
							).drop("InvoiceCalculationDate","BillingPeriodStartDate","BillingPeriodEndDate", "PatientClassificationOrder")
                            )

    pandasDf = df_AccuralMidwaySummary.toPandas()
    pandasDf = pandasDf.melt(id_vars=["Ship To","Billing Period Start Date","Billing Period End Date","Invoice Calculation Date"], var_name="Revenue Accrual Description", value_name="Revenue Accrual Total")

    pandasDf.insert(1, "Revenue Accrual Definition", "")

    pandasDf = pandasDf[["Ship To","Revenue Accrual Description","Revenue Accrual Definition","Revenue Accrual Total","Billing Period Start Date","Billing Period End Date","Invoice Calculation Date"]]

    df_AccuralMidwaySummary = spark.createDataFrame(pandasDf,schema)

    df_AccrualExcelSheet = add_Prefix_ColumnName(df_AccrualExcelSheet,"AccrualFile_")
    df_AccuralMidwaySummary = add_Prefix_ColumnName(df_AccuralMidwaySummary,"AddendumView_")

    df_result = (df_AccrualExcelSheet.alias("file").join(df_AccuralMidwaySummary.alias("goldzone"),(col("file.AccrualFile_Invoice Calculation Date") == col("goldzone.AddendumView_Invoice Calculation Date")) & (col("file.AccrualFile_Revenue Accrual Description") == col("goldzone.AddendumView_Revenue Accrual Description")),"FULL")
                                                    .withColumn("Comments",when(col("file.AccrualFile_Revenue Accrual Total") == col("goldzone.AddendumView_Revenue Accrual Total"),"Matched").otherwise("Unknown"))) 
    
    return df_result

In [0]:
def validateMidwayDetailSheet(df_AccrualExcelSheet,InvoiceDate,PromotionName,HashCols):

    df_AccrualFile = df_AccrualExcelSheet.withColumn("HashKey",sha2(concat_ws('', *HashCols),256))

    df_AddendumView = fetchInvoiceAddendumView(InvoiceDate,PromotionName)

    df_AccuralMidwayDetails = (df_AddendumView.filter(col("PatientClassificationOrder")==15).groupBy("Ship To","InvoiceCalculationDate","BillingPeriodStartDate","BillingPeriodEndDate").agg(
									coalesce(sum(when(col("PatientClassificationOrder")==15,col("Total"))),lit('0')).cast(LongType()).alias("Total MidwayPatient Fee"),
									date_format(col("BillingPeriodStartDate"),'MM-dd-yyyy').alias("Billing Period Start Date"),
									date_format(col("BillingPeriodEndDate"),'MM-dd-yyyy').alias("Billing Period End Date"),
									date_format(col("InvoiceCalculationDate"),'MM-dd-yyyy').alias("Invoice Calculation Date")
									).drop("InvoiceCalculationDate","BillingPeriodStartDate","BillingPeriodEndDate", "PatientClassificationOrder")
                                    .withColumn("HashKey",sha2(concat_ws('', *HashCols),256)))
    
    df_AccrualFile = add_Prefix_ColumnName(df_AccrualFile,"AccrualFile_")
    df_AccuralMidwayDetails = add_Prefix_ColumnName(df_AccuralMidwayDetails,"AddendumView_")

    df_result = (df_AccrualFile.alias("file").join(df_AccuralMidwayDetails.alias("goldzone"),(col("file.AccrualFile_Ship To") == col("goldzone.AddendumView_Ship To")),"FULL")
                                                    .withColumn("Comments",when(col("file.AccrualFile_HashKey") == col("goldzone.AddendumView_HashKey"),"Matched").otherwise("Unknown"))) 
    
    return df_result

In [0]:
def add_Prefix_ColumnName(df,prefix):
    new_cols = [col(col_name).alias(prefix+col_name) for col_name in df.columns]
    df_Source = df.select(new_cols)
    return df_Source

In [0]:
dst_TestExecutionResult = "/mnt/silver/TestExecutionResult"

def logIntoTestExecutionResultTable(log_dict):

    df_Source = (
        spark.createDataFrame(log_dict)
        .withColumn("CreatedBy", lit("ADB_AutomationTestCase"))
        .withColumn("UpdatedBy", lit("ADB_AutomationTestCase"))
        .withColumn("Createddate", current_timestamp())
        .withColumn("Updateddate", current_timestamp())
    )

    df_Source.write.format("delta").mode("append").save(dst_TestExecutionResult)

In [0]:
# def runAssert(condition, message, TestCaseName, InvoiceDate, MismatchedData):
#     try:
#         assert condition, message
#         print(f"\n \t TestCaseName: {TestCaseName}; TestCaseStatus: Passed")
#         log_dict = [{
#                     "TestCaseName" : TestCaseName,
#                     "TestCaseDescription" : TestCaseDetails[TestCaseName]["TestCaseDescription"],
#                     "InvoiceMonth" : str(InvoiceDate),
#                     "ValidationType" : "Post Validation Checks",
#                     "Status" : "Passed"
#                     }]
#         logIntoTestExecutionResultTable(log_dict)
#     except AssertionError as e:
#         assertion_errors.append(str(e))
#         print(f"\n \t TestCaseName:{TestCaseName}; TestCaseStatus: Failed")
#         log_dict = [{
#                     "TestCaseName" : TestCaseName,
#                     "TestCaseDescription" : TestCaseDetails[TestCaseName]["TestCaseDescription"],
#                     "TestCaseErrorMessage": message,
#                     "InvoiceMonth" : str(InvoiceDate),
#                     "ValidationType" : "Post Validation Checks",
#                     "MismatchedData":MismatchedData,
#                     "Status" : "Failed"
#                     }]
#         logIntoTestExecutionResultTable(log_dict)

## 1.Verify invoice addendum file gets created in storage account

In [0]:
def validateInvoiceFiles(InvoiceDate,TestCaseName):
    TestCaseStatus = []
    MismatchedDataList = []
    
    #Validate based on promotion
    df_promotion = spark.sql("SELECT DISTINCT PromotionName,InvoiceAddendumFilePath FROM promotion.dim_promotion where InvoiceAddendumFlag = true")

    invoicefiles_list = [{"PromotionName":row["PromotionName"],
                         "InvoiceAddendumFilePath": "/dbfs/mnt/"+ row["InvoiceAddendumFilePath"].split("?filePath=")[1].replace("<FilePath>","")} for row in df_promotion.collect()]

    for promotion in invoicefiles_list:
        print(f"Promotion Name:{promotion['PromotionName']}")
        invoice_filepath = promotion["InvoiceAddendumFilePath"]
        path_list = []
        FilePath = os.walk(f"{invoice_filepath}")
        for root,subdirs,files in FilePath:
            if root.endswith(InvoiceDate[:7]):
                if len(files) != 0:
                    FileName = files[0]
                else:
                    FileName = ''
                ShipToId = root.split('/')[-2]

                path_dict = {"ShipToAccountId":ShipToId,
                            "InvoiceFileName":FileName,
                            "InvoiceCalculationDate":InvoiceDate
                            }
                path_list.append(path_dict)

        schema = StructType([
        StructField("ShipToAccountId", StringType(), True),
        StructField("InvoiceFileName", StringType(), True),
        StructField("InvoiceCalculationDate", StringType(), True)
        ])
        df_storageaccount = spark.createDataFrame(path_list,schema)

        df_goldzone = spark.sql(f"select distinct ShipToAccountId,InvoiceFileName,InvoiceCalculationDate from goldzone.invoiceaddendum_v where InvoiceCalculationDate = '{InvoiceDate}' and PromotionName = '{promotion['PromotionName']}'")

        #Added Prefix in Dataframe Columns
        df_storageaccount = add_Prefix_ColumnName(df_storageaccount,"InvoiceFile_")
        df_goldzone = add_Prefix_ColumnName(df_goldzone,"AddendumView_")

        df_result = df_storageaccount.join(df_goldzone,(df_storageaccount.InvoiceFile_ShipToAccountId == df_goldzone.AddendumView_ShipToAccountId) & (df_storageaccount.InvoiceFile_InvoiceCalculationDate == df_goldzone.AddendumView_InvoiceCalculationDate),"FULL")\
                                    .withColumn("Comments",when(df_storageaccount.InvoiceFile_InvoiceFileName == df_goldzone.AddendumView_InvoiceFileName, "Matched")
                                                        .otherwise("Unknown"))
                                    
        ExceptionCount = df_result.filter("Comments == 'Unknown'").count()    
        MismatchedData = df_result.filter("Comments == 'Unknown'").toJSON().collect()

        if len(MismatchedData) != 0:
             MismatchedDataList.append(MismatchedData)
        
        try:    
            assert ExceptionCount == 0, TestCaseDetails[TestCaseName]["ErrorMessage"]
            print(f"\n \t TestCaseName: {TestCaseName}; TestCaseStatus: Passed")
        except AssertionError as e:
            assertion_errors.append(str(e))
            ErrorMessgae = f"{str(e)}:PromotionName:{promotion['PromotionName']}"
            TestCaseStatus.append(ErrorMessgae)
            print(f"\n \t TestCaseName:{TestCaseName}; TestCaseStatus: Failed")
        print(f"Comparison between Invoice File and InvoiceAddendum View")

        df_result.filter("Comments == 'Unknown'").display()
    
    if TestCaseStatus:
        log_dict = [{
                    "TestCaseName" : TestCaseName,
                    "TestCaseDescription" : TestCaseDetails[TestCaseName]["TestCaseDescription"],
                    "TestCaseErrorMessage": str(TestCaseStatus),
                    "InvoiceMonth" : str(InvoiceDate),
                    "ValidationType" : "Post Validation Checks",
                    "MismatchedData":str(MismatchedDataList),
                    "Status" : "Failed"
                    }]
        logIntoTestExecutionResultTable(log_dict)   
    else:
        log_dict = [{
                    "TestCaseName" : TestCaseName,
                    "TestCaseDescription" : TestCaseDetails[TestCaseName]["TestCaseDescription"],
                    "InvoiceMonth" : str(InvoiceDate),
                    "ValidationType" : "Post Validation Checks",
                    "Status" : "Passed"
                    }]
        logIntoTestExecutionResultTable(log_dict)


# 2. Validate Revenue Accrual File 

In [0]:
def validateRevenueAccrualFiles(InvoiceDate,TestCaseName):
    TestCaseStatus = []
    MismatchedDataList = []

    df_promotion = spark.sql(f"""SELECT DISTINCT PromotionName,PostInvoiceAccrualFilePath FROM promotion.dim_promotion where PostInvoiceAccrualFileFlag = true and cast('{InvoiceDate}' as Date) between EffectiveDate and TerminationDate""") 

    accrualfiles_list = [{"PromotionName":row["PromotionName"],
                        "AccrualFilePath": "/dbfs/mnt/"+ row["PostInvoiceAccrualFilePath"]+current_year+"-"+current_month+"/"+AccrualFileName} for row in df_promotion.collect()]
    for accrualfile in accrualfiles_list:
        try:
            print(f"Promotion Name:{accrualfile['PromotionName']}")

            #Copy Revenue Accrual file into tmp location
            if os.path.exists(accrualfile["AccrualFilePath"]):
                shutil.copyfile(accrualfile["AccrualFilePath"], temp_path)
            else:
                raise Exception(f"File does not exist:{accrualfile['AccrualFilePath']}")

            for filedetails in accrualsheet_dict:
                print(f"Excel Sheet Name:{filedetails['SheetName']}")

                #Read Revenue Accrual File from tmp location based on sheet name and convert it into pyspark dataframe
                sheets_dict = pd.read_excel(temp_path,sheet_name=filedetails['SheetName'],engine='openpyxl')

                df_AccrualExcelSheet = spark.createDataFrame(sheets_dict,filedetails['Schema'])

                if filedetails['SheetName'] == 'Accrual File Details':
                    df_result = validateAccrualDetailsSheet(df_AccrualExcelSheet,InvoiceDate,accrualfile['PromotionName'],filedetails['HashCols'])
                elif filedetails['SheetName'] == 'Accrual File Summary':
                    df_result = validateAccrualSummarySheet(df_AccrualExcelSheet,InvoiceDate,accrualfile['PromotionName'],filedetails['Schema'])
                elif filedetails['SheetName'] == 'Accrual File Midway Details':
                    df_result = validateMidwayDetailSheet(df_AccrualExcelSheet,InvoiceDate,accrualfile['PromotionName'],filedetails['HashCols'])
                elif filedetails['SheetName'] == 'Accrual File Midway Summary':
                    df_result = validateMidwaySummarySheet(df_AccrualExcelSheet,InvoiceDate,accrualfile['PromotionName'],filedetails['Schema'])

                ExceptionCount = df_result.filter("Comments == 'Unknown'").count()     
                MismatchedData = df_result.filter("Comments == 'Unknown'").toJSON().collect()

                if len(MismatchedData) != 0:
                    MismatchedDataList.append(MismatchedData)

                try:    
                    assert ExceptionCount == 0, TestCaseDetails[TestCaseName]["ErrorMessage"]
                    print(f"\n \t TestCaseName: {TestCaseName}; TestCaseStatus: Passed")
                except AssertionError as e:
                    assertion_errors.append(str(e))
                    ErrorMessgae = f"{str(e)}:PromotionName:{accrualfile['PromotionName']},SheetName:{filedetails['SheetName']}"
                    TestCaseStatus.append(ErrorMessgae)
                    print(f"\n \t TestCaseName:{TestCaseName}; TestCaseStatus: Failed")
                print(f"Comparison between Revenue Accrual File and InvoiceAddendum View")

                df_result.filter("Comments == 'Unknown'").display()

            #Remove the file from tmp location after Validation completed for all the sheet in a file
            os.remove(temp_path)
        except Exception as e:
            print(traceback.print_exc())
            assertion_errors.append(str(e))
            ErrorMessgae = f"{str(e)}:PromotionName{accrualfile['PromotionName']}"
            print(ErrorMessgae)
            TestCaseStatus.append(ErrorMessgae)
    if TestCaseStatus:
        log_dict = [{
                    "TestCaseName" : TestCaseName,
                    "TestCaseDescription" : TestCaseDetails[TestCaseName]["TestCaseDescription"],
                    "TestCaseErrorMessage": str(TestCaseStatus),
                    "InvoiceMonth" : str(InvoiceDate),
                    "ValidationType" : "Post Validation Checks",
                    "MismatchedData": str(MismatchedDataList),
                    "Status" : "Failed"
                    }]
        # print(log_dict)
        logIntoTestExecutionResultTable(log_dict)   
    else:
        log_dict = [{
                    "TestCaseName" : TestCaseName,
                    "TestCaseDescription" : TestCaseDetails[TestCaseName]["TestCaseDescription"],
                    "InvoiceMonth" : str(InvoiceDate),
                    "ValidationType" : "Post Validation Checks",
                    "Status" : "Passed"
                    }]
        logIntoTestExecutionResultTable(log_dict)

In [0]:
validateInvoiceFiles(str(InvoiceDate),"validateInvoiceFiles")
validateRevenueAccrualFiles(InvoiceDate,"validateRevenueAccrualFiles")

In [0]:
# Check if there were any Failures in Test Cases
if assertion_errors:
    raise Exception("Failed Test Cases: " + "; ".join(set(assertion_errors)))
print("All Test Cases are passed.")